# Nível 1 — Dados e primeira análise com LLM

Triagem de PLD sobre `dados/dados_nivel_1.json` (20 operações, 6 clientes), em duas partes:

- **Parte A — pandas:** diagnóstico e limpeza dos dados, normalização para BRL, agregações e duas regras determinísticas, com validação.
- **Parte B — LLM:** parecer estruturado para um cliente sinalizado, com validação de schema, medição de tokens/latência e comparação de dois prompts.

> **Princípio que rege o notebook:** somar, contar, tirar mediana e comparar com limite é **cálculo — feito em pandas**. A LLM recebe os números prontos e entra **apenas para interpretar e redigir**.

## Parte A — Tratamento e regras

### A.1 Carga e diagnóstico de qualidade

O enunciado avisa que os dados vêm de um sistema legado e não estão limpos. Antes de tratar, **medir**: procuro registros duplicados (linha inteira e `id` repetido com conteúdo divergente), datas nulas, valores inválidos e moedas misturadas — os defeitos clássicos de extração legada.

In [1]:
import json

import pandas as pd

with open("../dados/dados_nivel_1.json", encoding="utf-8") as f:
    raw = json.load(f)

TAXA_USD_BRL = raw["taxa_cambio_usd_brl"]
df = pd.DataFrame(raw["operacoes"])

print(f"{len(df)} operações | {df['cliente_id'].nunique()} clientes | taxa USD→BRL fixa = {TAXA_USD_BRL}")
df.head()

20 operações | 6 clientes | taxa USD→BRL fixa = 5.4


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [2]:
diagnostico = pd.Series({
    "registros duplicados (linha inteira idêntica)": df.duplicated().sum(),
    "mesmo id com conteúdo divergente": (df.duplicated("id", keep=False) & ~df.duplicated(keep=False)).sum(),
    "datas nulas": df["data"].isna().sum(),
    "valores nulos ou <= 0": (df["valor"].isna() | (df["valor"] <= 0)).sum(),
    "operações em moeda estrangeira": (df["moeda"] != "BRL").sum(),
}, name="ocorrências").to_frame()
diagnostico

,ocorrências
registros duplicados (linha inteira idêntica),1
mesmo id com conteúdo divergente,0
datas nulas,1
valores nulos ou <= 0,0
operações em moeda estrangeira,1


In [3]:
print("Registro duplicado (OP-0007 aparece duas vezes, idêntica em todos os campos):")
display(df[df.duplicated(keep=False)])

print("Data nula:")
display(df[df["data"].isna()])

print("Moeda estrangeira:")
display(df[df["moeda"] != "BRL"])

Registro duplicado (OP-0007 aparece duas vezes, idêntica em todos os campos):


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


Data nula:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


Moeda estrangeira:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional


### A.2 Decisões de limpeza

Três problemas plantados, três decisões (o raciocínio completo está em `docs/DECISOES.md`):

| Problema | Decisão | Justificativa |
|---|---|---|
| **OP-0007 duplicada** (2× idêntica) | Remover a cópia | Mesmo `id` e todos os campos iguais → reprocessamento do legado, não duas operações reais. Manter a cópia **muda o resultado da Regra 1** (demonstro na validação, A.5). |
| **OP-0017 sem data** ("data nao capturada pelo sistema") | Manter nos volumes; excluir **só** das regras que dependem de data | A operação aconteceu — R$ 4.300 de depósito **em espécie** (justamente o canal mais sensível em PLD) devem contar no volume. Só a data é desconhecida. |
| **OP-0013 em USD** (US$ 12.000) | Converter com a taxa fixa do arquivo (5.4) em coluna nova `valor_brl` | Comparar limites exige moeda única. Preservo `valor`/`moeda` originais para auditoria. Sem conversão, essa operação escaparia da Regra 2. |

In [4]:
df_limpo = df.drop_duplicates().copy()

# valor_brl: moeda unica para todas as comparacoes; original preservado para auditoria
df_limpo["valor_brl"] = df_limpo["valor"].where(df_limpo["moeda"] == "BRL", df_limpo["valor"] * TAXA_USD_BRL)
df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")

print(f"{len(df)} operações → {len(df_limpo)} após remover duplicata exata")
df_limpo.loc[df_limpo["moeda"] == "USD", ["id", "cliente_id", "valor", "moeda", "valor_brl"]]

20 operações → 19 após remover duplicata exata


,id,cliente_id,valor,moeda,valor_brl
13,OP-0013,CLI-A-4,12000,USD,64800


### A.3 Agregações

In [5]:
volume_por_cliente = (
    df_limpo.groupby("cliente_id")
    .agg(qtd_operacoes=("id", "size"), volume_total_brl=("valor_brl", "sum"))
    .sort_values("volume_total_brl", ascending=False)
)
volume_por_cliente

,qtd_operacoes,volume_total_brl
cliente_id,,
CLI-A-4,4,79500
CLI-A-1,4,57500
CLI-A-2,2,52900
CLI-A-3,3,48500
CLI-A-5,4,16900
CLI-A-6,2,10200


In [6]:
operacoes_por_canal = df_limpo["canal"].value_counts().rename_axis("canal").to_frame("qtd_operacoes")
operacoes_por_canal

,qtd_operacoes
canal,
pix,8
ted,5
boleto,3
cartao,2
especie,1


### A.4 Regras determinísticas

**Regra 1 — Fracionamento (*smurfing*):** cliente com **3+ operações na mesma data** somando **mais de R$ 50.000**, sem que **nenhuma** atinja R$ 20.000 — o padrão de quebrar um valor grande em vários pequenos para escapar de limites de reporte.

**Regra 2 — Valor atípico:** operação **acima de 5× a mediana** das operações do próprio cliente (em BRL), aplicada apenas a clientes com **4+ operações** (mediana de poucas operações não é referência).

Interpretações registradas em `DECISOES.md`: a mediana inclui a própria operação avaliada (no Nível 1 o resultado é idêntico ao excluí-la), e operações sem data ficam fora apenas da Regra 1 — seguem contando para volume e mediana.

As flags entram **no DataFrame**: `flag_fracionamento` marca cada operação do grupo (cliente, dia) enquadrado; `flag_valor_atipico` marca a operação individual que dispara a Regra 2.

In [7]:
LIMIAR_SOMA_DIA = 50_000       # soma diaria que caracteriza fracionamento
LIMIAR_OP_ISOLADA = 20_000     # nenhuma operacao isolada atinge este valor
MIN_OPS_DIA = 3                # minimo de operacoes no mesmo dia
FATOR_ATIPICO = 5              # multiplo da mediana do cliente
MIN_OPS_CLIENTE = 4            # minimo de operacoes para a Regra 2


def aplicar_regra_fracionamento(ops: pd.DataFrame) -> pd.DataFrame:
    """Regra 1: marca operações de grupos (cliente, dia) com 3+ ops, soma > 50k e todas < 20k."""
    grupos = (
        ops.dropna(subset=["data"])
        .groupby(["cliente_id", "data"])["valor_brl"]
        .agg(["size", "sum", "max"])
    )
    dias_suspeitos = grupos[
        (grupos["size"] >= MIN_OPS_DIA)
        & (grupos["sum"] > LIMIAR_SOMA_DIA)
        & (grupos["max"] < LIMIAR_OP_ISOLADA)
    ].index
    flag = ops.set_index(["cliente_id", "data"]).index.isin(dias_suspeitos)
    return ops.assign(flag_fracionamento=flag)


def aplicar_regra_valor_atipico(ops: pd.DataFrame) -> pd.DataFrame:
    """Regra 2: marca a operação acima de 5x a mediana do cliente (clientes com 4+ ops)."""
    qtd_ops_cliente = ops.groupby("cliente_id")["id"].transform("size")
    mediana_cliente = ops.groupby("cliente_id")["valor_brl"].transform("median")
    flag = (qtd_ops_cliente >= MIN_OPS_CLIENTE) & (ops["valor_brl"] > FATOR_ATIPICO * mediana_cliente)
    return ops.assign(mediana_cliente=mediana_cliente, flag_valor_atipico=flag)


df_limpo = aplicar_regra_valor_atipico(aplicar_regra_fracionamento(df_limpo))

print("Operações sinalizadas pela Regra 1 (fracionamento):")
display(df_limpo.loc[df_limpo["flag_fracionamento"], ["id", "cliente_id", "data", "valor_brl", "canal", "contraparte"]])

print("Operações sinalizadas pela Regra 2 (valor atípico):")
display(df_limpo.loc[df_limpo["flag_valor_atipico"], ["id", "cliente_id", "valor", "moeda", "valor_brl", "mediana_cliente"]])

Operações sinalizadas pela Regra 1 (fracionamento):


,id,cliente_id,data,valor_brl,canal,contraparte
0,OP-0001,CLI-A-1,2026-03-09,18100,pix,Alfa Comercio LTDA
1,OP-0002,CLI-A-1,2026-03-09,17300,pix,Alfa Comercio LTDA
2,OP-0003,CLI-A-1,2026-03-09,18800,ted,Beta Servicos ME


Operações sinalizadas pela Regra 2 (valor atípico):


,id,cliente_id,valor,moeda,valor_brl,mediana_cliente
13,OP-0013,CLI-A-4,12000,USD,64800,5450.0


In [8]:
resumo_clientes = df_limpo.groupby("cliente_id").agg(
    operacoes=("id", "size"),
    volume_brl=("valor_brl", "sum"),
    fracionamento=("flag_fracionamento", "any"),
    valor_atipico=("flag_valor_atipico", "any"),
)
resumo_clientes["sinalizado"] = resumo_clientes["fracionamento"] | resumo_clientes["valor_atipico"]
resumo_clientes

,operacoes,volume_brl,fracionamento,valor_atipico,sinalizado
cliente_id,,,,,
CLI-A-1,4,57500,True,False,True
CLI-A-2,2,52900,False,False,False
CLI-A-3,3,48500,False,False,False
CLI-A-4,4,79500,False,True,True
CLI-A-5,4,16900,False,False,False
CLI-A-6,2,10200,False,False,False


### A.5 Validação da Regra 1

Validação em dois atos:

1. **Captura quem deve e poupa o caso parecido.** CLI-A-1 e CLI-A-3 têm o mesmo desenho — 3 transferências no mesmo dia, todas abaixo de R$ 20 mil. A diferença é a soma: CLI-A-1 ultrapassa R$ 50 mil (54.200) e é sinalizado; CLI-A-3 fica abaixo (48.500) e **não** é.
2. **A limpeza importa.** Se a duplicata OP-0007 não fosse removida, o CLI-A-3 "somaria" R$ 65.700 e viraria um **falso positivo** — a regra certa sobre o dado errado produz a resposta errada.

In [9]:
validacao = (
    df_limpo.dropna(subset=["data"])
    .groupby(["cliente_id", "data"])["valor_brl"]
    .agg(qtd_ops="size", soma_dia="sum", maior_op="max")
    .query("qtd_ops >= @MIN_OPS_DIA")
    .assign(
        soma_ultrapassa_50k=lambda g: g["soma_dia"] > LIMIAR_SOMA_DIA,
        todas_abaixo_20k=lambda g: g["maior_op"] < LIMIAR_OP_ISOLADA,
    )
)
validacao["enquadra_regra_1"] = validacao["soma_ultrapassa_50k"] & validacao["todas_abaixo_20k"]
validacao

,,qtd_ops,soma_dia,maior_op,soma_ultrapassa_50k,todas_abaixo_20k,enquadra_regra_1
cliente_id,data,,,,,,
CLI-A-1,2026-03-09,3,54200,18800,True,True,True
CLI-A-3,2026-03-05,3,48500,17200,False,True,False


In [10]:
# Contrafactual: a mesma regra aplicada ao dado SUJO (duplicata mantida)
df_sujo = df.copy()
df_sujo["valor_brl"] = df_sujo["valor"].where(df_sujo["moeda"] == "BRL", df_sujo["valor"] * TAXA_USD_BRL)
df_sujo["data"] = pd.to_datetime(df_sujo["data"], errors="coerce")

flags_sujas = aplicar_regra_fracionamento(df_sujo)
comparativo = pd.DataFrame({
    "dado sujo (com duplicata)": flags_sujas.loc[flags_sujas["flag_fracionamento"]]
        .groupby("cliente_id")["valor_brl"].sum(),
    "dado limpo": df_limpo.loc[df_limpo["flag_fracionamento"]]
        .groupby("cliente_id")["valor_brl"].sum(),
})
print("Soma diária dos clientes sinalizados pela Regra 1 em cada cenário (NaN = não sinalizado):")
comparativo

Soma diária dos clientes sinalizados pela Regra 1 em cada cenário (NaN = não sinalizado):


,dado sujo (com duplicata),dado limpo
cliente_id,,
CLI-A-1,54200,54200.0
CLI-A-3,65700,NaN


### Síntese da Parte A

| Cliente | Situação | Evidência |
|---|---|---|
| **CLI-A-1** | 🚩 **Fracionamento** (Regra 1) | 3 transferências em 09/03 para 2 contrapartes, somando R$ 54.200 — todas entre R$ 17,3 mil e R$ 18,8 mil, logo abaixo do limite de R$ 20 mil |
| **CLI-A-4** | 🚩 **Valor atípico** (Regra 2) | OP-0013: remessa internacional de US$ 12.000 (R$ 64.800) = **11,9×** a mediana do cliente (R$ 5.450) — só detectável após a conversão cambial |
| CLI-A-3 | ✅ Falso positivo **evitado** | Padrão parecido com CLI-A-1, mas soma R$ 48.500 (< 50 mil); com a duplicata OP-0007, seria sinalizado indevidamente |
| CLI-A-2, CLI-A-5, CLI-A-6 | ✅ Sem sinalização | Nenhuma regra disparada |

A Parte B toma o **CLI-A-1** — o caso de fracionamento — e pede à LLM o parecer de risco, com os números calculados aqui.

## Parte B — Análise com LLM

Cliente escolhido: **CLI-A-1**, sinalizado por fracionamento na Parte A.

O fluxo respeita a separação de papéis: os números que alimentam o prompt saem do
DataFrame (célula abaixo) — a LLM recebe tudo **pronto** e só interpreta e redige.
O provedor é intercambiável via `.env` (endpoint compatível com OpenAI; aqui, Gemini).

Nesta parte: resposta em **estrutura validada** (`nivel_risco`, `tipologia_suspeita`,
`red_flags`, `justificativa`), **tratamento de resposta malformada** com re-tentativa
corretiva, registro de **tokens e tempo**, e **duas versões de prompt** comparadas.

In [11]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("..") / ".env")
llm = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
MODELO = os.environ["LLM_MODEL"]
print(f"Provedor configurado via .env | modelo: {MODELO}")

Provedor configurado via .env | modelo: openai/gpt-oss-120b


In [12]:
# Dossiê do CLI-A-1 calculado EM PANDAS — a LLM não faz nenhuma conta
ops_cli = df_limpo[df_limpo["cliente_id"] == "CLI-A-1"]
dia_suspeito = ops_cli.loc[ops_cli["flag_fracionamento"], "data"].iloc[0]
ops_dia = ops_cli[ops_cli["data"] == dia_suspeito]

dossie = {
    "cliente_id": "CLI-A-1",
    "qtd_operacoes_no_mes": int(len(ops_cli)),
    "volume_total_brl": float(ops_cli["valor_brl"].sum()),
    "mediana_das_operacoes_brl": float(ops_cli["valor_brl"].median()),
    "dia_sinalizado_pela_regra_de_fracionamento": str(dia_suspeito.date()),
    "operacoes_do_dia_sinalizado": ops_dia[["id", "valor_brl", "canal", "tipo", "contraparte"]]
        .to_dict(orient="records"),
    "soma_do_dia_sinalizado_brl": float(ops_dia["valor_brl"].sum()),
    "maior_operacao_do_dia_brl": float(ops_dia["valor_brl"].max()),
    "limiares_da_regra": {"soma_diaria": 50_000, "operacao_isolada": 20_000, "min_operacoes": 3},
}
dossie

{'cliente_id': 'CLI-A-1',
 'qtd_operacoes_no_mes': 4,
 'volume_total_brl': 57500.0,
 'mediana_das_operacoes_brl': 17700.0,
 'dia_sinalizado_pela_regra_de_fracionamento': '2026-03-09',
 'operacoes_do_dia_sinalizado': [{'id': 'OP-0001',
   'valor_brl': 18100,
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0002',
   'valor_brl': 17300,
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0003',
   'valor_brl': 18800,
   'canal': 'ted',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Beta Servicos ME'}],
 'soma_do_dia_sinalizado_brl': 54200.0,
 'maior_operacao_do_dia_brl': 18800.0,
 'limiares_da_regra': {'soma_diaria': 50000,
  'operacao_isolada': 20000,
  'min_operacoes': 3}}

In [13]:
import json

CAMPOS_OBRIGATORIOS = {"nivel_risco", "tipologia_suspeita", "red_flags", "justificativa"}
NIVEIS_VALIDOS = {"baixo", "medio", "alto"}


def validar_parecer(texto: str):
    """Valida a estrutura do parecer. Devolve (dict, None) ou (None, motivo do erro)."""
    texto = (texto or "").strip()
    if texto.startswith("```"):
        texto = texto[texto.find("{"): texto.rfind("}") + 1]
    try:
        dados = json.loads(texto)
    except json.JSONDecodeError as exc:
        return None, f"JSON inválido: {exc}"
    faltando = CAMPOS_OBRIGATORIOS - dados.keys()
    if faltando:
        return None, f"campos ausentes: {sorted(faltando)}"
    if dados["nivel_risco"] not in NIVEIS_VALIDOS:
        return None, f"nivel_risco fora do domínio: {dados['nivel_risco']!r}"
    if not isinstance(dados["red_flags"], list):
        return None, "red_flags deve ser lista"
    return dados, None


def pedir_parecer(prompt: str, max_correcoes: int = 2):
    """Chama a LLM medindo tokens e tempo; se a resposta vier malformada,
    re-envia UMA instrução corretiva (até max_correcoes vezes)."""
    mensagens = [{"role": "user", "content": prompt}]
    total = {"tokens_entrada": 0, "tokens_saida": 0, "chamadas": 0}
    inicio = time.perf_counter()

    for _ in range(1 + max_correcoes):
        resposta = llm.chat.completions.create(model=MODELO, messages=mensagens)
        total["chamadas"] += 1
        total["tokens_entrada"] += resposta.usage.prompt_tokens
        total["tokens_saida"] += resposta.usage.completion_tokens
        texto = resposta.choices[0].message.content
        parecer, erro = validar_parecer(texto)
        if parecer is not None:
            break
        mensagens += [
            {"role": "assistant", "content": texto},
            {"role": "user", "content": f"Sua resposta falhou na validação ({erro}). "
                                        "Reenvie SOMENTE o JSON corrigido, sem outro texto."},
        ]
    total["latencia_s"] = round(time.perf_counter() - inicio, 2)
    return parecer, erro, total


# Demonstração do validador nos casos de defeito (sem gastar cota):
for quebrado in ['não é json', '{"nivel_risco": "altissimo", "tipologia_suspeita": "x", "red_flags": [], "justificativa": "y"}', '{"nivel_risco": "alto"}']:
    _, motivo = validar_parecer(quebrado)
    print(f"rejeitado -> {motivo}")

rejeitado -> JSON inválido: Expecting value: line 1 column 1 (char 0)
rejeitado -> nivel_risco fora do domínio: 'altissimo'
rejeitado -> campos ausentes: ['justificativa', 'red_flags', 'tipologia_suspeita']


### Prompt v1 — direto

Pedido mínimo: os dados e o formato de saída, sem persona nem critérios. Serve de linha de base.

In [14]:
prompt_v1 = f"""Analise as movimentações do cliente abaixo e produza um parecer de risco de
lavagem de dinheiro. Responda somente com um JSON com os campos: nivel_risco
(baixo/medio/alto), tipologia_suspeita, red_flags (lista), justificativa.

Dados do cliente (já calculados):
{json.dumps(dossie, ensure_ascii=False, indent=2)}"""

parecer_v1, erro_v1, metricas_v1 = pedir_parecer(prompt_v1)
print(json.dumps(parecer_v1, ensure_ascii=False, indent=2))
print("\nmétricas:", metricas_v1)

{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Estruturação de Operações (Smurfing)",
  "red_flags": [
    "Concentração de três transferências em um único dia (2026-03-09) ultrapassando o limite diário de R$ 50.000,00",
    "Valor total do dia próximo ao teto da regra de soma diária (R$ 54.200,00)",
    "Operações realizadas em canais de alta velocidade e baixo custo (PIX) combinados com TED",
    "Repetição de contraparte (Alfa Comercio LTDA) em duas das três transferências",
    "Valor individual próximo ao limite de operação isolada (R$ 20.000,00) – 18.800,00",
    "Número reduzido de operações no mês (4) com volume elevado (R$ 57.500,00), indicando alta concentração de recursos",
    "Padrão que sugere tentativa de fracionamento para evitar detecção automática"
  ],
  "justificativa": "A regra de fracionamento foi acionada porque, no dia 09/03/2026, o cliente realizou três transferências que somaram R$ 54.200,00, ultrapassando o limite de R$ 50.000,00 definido para a soma diá

### Prompt v2 — persona + critérios explícitos

Mesmos dados, mas com papel definido (analista de PLD), vocabulário da área, critérios de
decisão para cada nível de risco e a instrução de citar os números do dossiê na justificativa.

In [15]:
prompt_v2 = f"""Você é um analista sênior de Prevenção à Lavagem de Dinheiro (PLD) de um banco
brasileiro, redigindo um parecer para a mesa de triagem.

Contexto da sinalização: a regra determinística de fracionamento marca clientes com 3 ou mais
operações no mesmo dia somando acima de R$ 50.000, todas individualmente abaixo de R$ 20.000
(padrão conhecido como smurfing/estruturação: fatiar valores para escapar de limites de reporte).

Todos os números abaixo já foram calculados pelo sistema — não recalcule nada; seu papel é
interpretar o padrão e redigir o parecer.

Dossiê do cliente:
{json.dumps(dossie, ensure_ascii=False, indent=2)}

Critérios para o nível de risco:
- alto: padrão compatível com tipologia conhecida de lavagem (ex.: estruturação deliberada);
- medio: desvio relevante do comportamento, mas com possível explicação legítima;
- baixo: movimentação compatível com o perfil.

Responda SOMENTE com um JSON válido, sem markdown, com os campos:
nivel_risco (baixo/medio/alto), tipologia_suspeita, red_flags (lista de indícios, citando
valores e datas do dossiê), justificativa (parágrafo citando explicitamente os números)."""

parecer_v2, erro_v2, metricas_v2 = pedir_parecer(prompt_v2)
print(json.dumps(parecer_v2, ensure_ascii=False, indent=2))
print("\nmétricas:", metricas_v2)

{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Estruturação (smurfing)",
  "red_flags": [
    "Três operações de transferência enviadas em 2026-03-09 somando R$54.200, acima do limiar de R$50.000 da regra de fracionamento",
    "Todas as três operações individuais estão abaixo do limite de R$20.000 (R$18.100, R$17.300 e R$18.800)",
    "Concentração de 3 das 4 operações do mês em um único dia, representando 94,2% do volume mensal (R$57.500)",
    "Uso de diferentes canais (PIX e TED) para as operações, dificultando a rastreabilidade por canal único"
  ],
  "justificativa": "No dia 09/03/2026 o cliente realizou três transferências enviadas (IDs OP-0001, OP-0002 e OP-0003) pelos canais PIX e TED, com valores de R$18.100, R$17.300 e R$18.800, respectivamente, totalizando R$54.200. Cada operação está abaixo do limite individual de R$20.000, mas a soma diária excede o limiar de R$50.000 estabelecido para a regra de fracionamento, configurando o padrão clássico de smurfing. Além disso, 

In [16]:
import pandas as pd

comparacao_prompts = pd.DataFrame([
    {"prompt": "v1 — direto", **metricas_v1,
     "nivel_risco": parecer_v1["nivel_risco"] if parecer_v1 else "FALHOU",
     "qtd_red_flags": len(parecer_v1["red_flags"]) if parecer_v1 else None},
    {"prompt": "v2 — persona + critérios", **metricas_v2,
     "nivel_risco": parecer_v2["nivel_risco"] if parecer_v2 else "FALHOU",
     "qtd_red_flags": len(parecer_v2["red_flags"]) if parecer_v2 else None},
]).set_index("prompt")
comparacao_prompts

,tokens_entrada,tokens_saida,chamadas,latencia_s,nivel_risco,qtd_red_flags
prompt,,,,,,
v1 — direto,478,740,1,3.90,alto,7
v2 — persona + critérios,685,646,1,1.64,alto,4


### Comparação e conclusão da Parte B

Os dois prompts devolveram JSON **válido na primeira chamada** e concordaram no essencial:
nível **alto** e tipologia **estruturação (smurfing)**. As diferenças estão na qualidade do parecer:

- **Precisão e ancoragem:** o v2 cita os IDs das operações (OP-0001/0002/0003) e os valores exatos
  (R$ 18.100, 17.300, 18.800) do dossiê. O v1 produziu mais red flags (7 × 4), mas com ruído: trata o
  limiar da regra como "limite diário" do banco e descreve R$ 54.200 como "próximo ao teto" — quando o
  ponto é justamente que a soma **ultrapassa** o limiar de R$ 50.000. Mais flags ≠ melhor parecer.
- **Custo e latência:** o v2 gasta mais tokens de entrada (prompt maior: 685 × 478) e menos de saída
  (646 × 740), com latência menor nesta amostra — em lote, saída é o token mais caro, então o prompt
  mais rico praticamente se paga.
- **Achado honesto (limite da abordagem):** mesmo instruído a não calcular, o v2 derivou um número
  novo — "94,2% do volume mensal concentrado no dia" (54.200 / 57.500). O valor está **correto**, mas é
  exatamente o comportamento que a separação regra × LLM quer conter: em produção, o validador deveria
  conferir todo número do parecer contra o dossiê (ou o prompt deveria fornecer também os percentuais
  pré-calculados). Registrado como limitação em `docs/DECISOES.md`.

**Conclusão:** persona + critérios explícitos + instrução de citar números compraram precisão e
verificabilidade ao custo de ~40% mais tokens de entrada — em PLD, onde parecer impreciso vira
retrabalho de mesa (ou risco regulatório), a troca vale a pena. O v2 é o formato que o agente do
Nível 2 herda.